In [7]:
%load_ext autoreload
%autoreload 2

import mlflow
from catboost import CatBoostRegressor
from data_prep import load_data, prepare_data
from feature_eng import features, cat_features, add_feature_wear
from train import train_model, log_experiment

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
path = "../data/kufar_auto_v2.csv"
df = load_data(path)
df = prepare_data(df)
df = add_feature_wear(df)

df.info()

Отсеяно по неправдоподобной цене: 43 из 44409
Отсеяно по нереальному пробегу: 448 из 44366
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43918 entries, 0 to 43917
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ad_id       43918 non-null  int64  
 1   subject     43918 non-null  object 
 2   price_usd   43918 non-null  float64
 3   regdate     43918 non-null  int64  
 4   mileage     43918 non-null  int64  
 5   engine      43918 non-null  object 
 6   capacity    43918 non-null  float64
 7   gearbox     43918 non-null  object 
 8   body_type   43918 non-null  object 
 9   drive       43918 non-null  object 
 10  seats       43918 non-null  int64  
 11  condition   43918 non-null  object 
 12  brand       43918 non-null  object 
 13  model       43918 non-null  object 
 14  generation  43918 non-null  object 
 15  ad_link     43918 non-null  object 
 16  list_time   43918 non-null  object 
 17  wear        4391

In [9]:
X = df[features]
y = df["price_usd"]

In [10]:
X.head()

,regdate,mileage,engine,capacity,gearbox,body_type,drive,brand,model,wear
0,2001,458000,Дизель,2.0,Механика,Минивэн,Передний,Citroen,Evasion,17615.4
1,2011,395000,Бензин,3.6,Автоматическая,Внедорожник,Полный,Porsche,Cayenne,24687.5
2,1998,252000,Бензин,1.4,Механика,Хэтчбек,Передний,Mercedes-Benz,A-Класс,8689.7
3,1997,350000,Бензин,1.8,Механика,Универсал,Передний,Ford,Mondeo,11666.7
4,1992,400000,Дизель,1.9,Механика,Фургон,Передний,Volkswagen,Transporter,11428.6


In [11]:
y.head()

0     4500.00
1    23000.00
2     3700.00
3      649.52
4     4800.00
Name: price_usd, dtype: float64

In [12]:
mlflow.set_experiment("catboost-kufar-auto")
log_params = {
    "iterations": 900,
    "depth": 5,
    "learning_rate": 0.02,
}
unlog_params = {
    "random_state": 42,
    "verbose": False,
}
model_params = log_params | unlog_params
#holdout
mode = "cv"

with mlflow.start_run():

    model = CatBoostRegressor(**model_params, cat_features=cat_features)

    results, n_rows = train_model(
        model=model,
        X=X,
        y=y,
        mode=mode,
        test_size=0.2,
        random_state=42,
    )

    log_experiment(results, model, log_params, mode, features, path, n_rows)

    

Доступные метрики: ['rmse', 'nrmse', 'r2', 'mape']
RMSE:  2972.17
NRMSE: 31.82%
R2:    0.8979
MAPE:    33.91
dict_items([('iterations', 900), ('learning_rate', 0.02), ('depth', 5), ('loss_function', 'RMSE'), ('verbose', False), ('random_state', 42), ('cat_features', ['engine', 'gearbox', 'body_type', 'drive', 'brand', 'model'])])


In [13]:
model.get_feature_importance()

array([52.61106847,  2.14273315,  3.02622064, 18.95744798,  3.15962927,
        3.7898581 ,  6.15299806,  7.34171114,  1.56135652,  1.25697667])